In [1]:
import os
import glob
import sys
from datetime import datetime
import matplotlib.pyplot as plt
import random
import cv2
import numbers
import wandb
import warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from typing import Union
from dataclasses import dataclass, asdict
from torchsummary import summary
import math
from einops import rearrange, repeat, einsum

def count_params(model):
    """ Count model trainable parameters """
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
print(f'Number of available cpu: {os.cpu_count()}')

Using device: cuda
Number of available cpu: 80


In [ ]:
# Install monarch-attention package
!git clone https://github.com/cjyaras/monarch-attention
!pip install ./monarch-attention

In [ ]:
!pip install einops
!pip install torchsummary
!pip install hdf5storage h5py
!pip install albumentations
!pip install pytorch_msssim

# Data Pipeline

In [ ]:
"""
Prepare training and validation datasets, dataloaders, and model architecture. Set up training loop with visualization and checkpointing.
"""

# Mobile Sketch2Image (MS2I) Network Implementation

## Common Blocks

In [2]:
class UpSample(nn.Module):
    """ UpSampling block using PixelShuffle """
    def __init__(self, filters=64):
        super().__init__()
        self.conv = nn.Conv2d(filters, filters * 2, kernel_size=1, stride=1, padding=0, bias=True)
        self.pixel_shuffle = nn.PixelShuffle(upscale_factor=2)

    def forward(self, x):
        x = self.conv(x)
        x = self.pixel_shuffle(x)
        return x

## DownSampling block
class DownSample(nn.Module):
    """ DownSampling block using PixelUnshuffle """
    def __init__(self, filters=64):
        super().__init__()
        self.conv = nn.Conv2d(filters, filters // 2, kernel_size=1, stride=1, padding=0, bias=True)
        self.pixel_unshuffle = nn.PixelUnshuffle(downscale_factor=2)

    def forward(self, x):
        """ SHAPE (B, C, H, W) -> SHAPE (B, C/4, H/2, W/2) """
        x = self.conv(x)
        x = self.pixel_unshuffle(x)
        return x

class ConvDown(nn.Module):
    """ DownSampling block using strided convolution """
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=2, padding=1, bias=True)

    def forward(self, x):
        x = self.conv(x)
        return x

class ConvUp(nn.Module):
    """ UpSampling block using Upsample + convolution """
    def __init__(self, in_channels, out_channels, out_shape=None, scale_factor=None):
        super().__init__()
        assert (out_shape is not None) ^ (scale_factor is not None), "Either out_shape or scale_factor must be provided, but not both."
        if out_shape:
            self.out_shape = out_shape
            self.upsample = nn.Upsample(out_shape, mode='bilinear', align_corners=False)
        else:
            self.upsample = nn.Upsample(scale_factor=scale_factor, mode='bilinear', align_corners=False)
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=True)

    def forward(self, x):
        x = self.upsample(x)
        x = self.conv(x)
        return x

def shape_estimation(h, w, kernel_size=3, stride=1, padding=1):
    """ Estimate the output shape after a convolutional layer. """
    out_h = (h + 2*padding - kernel_size) // stride + 1
    out_w = (w + 2*padding - kernel_size) // stride + 1
    return out_h, out_w

In [3]:
# Basic blocks
class DConvBlock(nn.Module):
    """ Custom Depthwise Convolution Block """
    def __init__(self, inshape, dim=64, expansion_factor=1.0, bias=False):
        super().__init__()
        hidden_features = int(dim*expansion_factor)
        self.conv = nn.Conv2d(inshape, hidden_features, kernel_size=1, bias=bias)
        self.depthwise = nn.Conv2d(hidden_features, hidden_features, kernel_size=3, stride=1, padding=1, groups=hidden_features, bias=bias)

    def forward(self, x):
        x = self.conv(x)
        x = self.depthwise(x)
        return x

# Custom LayerNormalization
class BiasFree_LayerNorm(nn.Module):
    """ Bias-Free Layer Normalization """
    def __init__(self, normalized_shape):
        super().__init__()
        if isinstance(normalized_shape, numbers.Integral):
            normalized_shape = (normalized_shape,)
        normalized_shape = torch.Size(normalized_shape)

        assert len(normalized_shape) == 1

        self.weight = nn.Parameter(torch.ones(normalized_shape))
        self.normalized_shape = normalized_shape

    def forward(self, x):
        x = x.contiguous() 
        sigma = x.var(-1, keepdim=True, unbiased=False)
        return x / torch.sqrt(sigma+1e-5) * self.weight

class WithBias_LayerNorm(nn.Module):
    """ With-Bias Layer Normalization """
    def __init__(self, normalized_shape):
        super().__init__()
        if isinstance(normalized_shape, numbers.Integral):
            normalized_shape = (normalized_shape,)
        normalized_shape = torch.Size(normalized_shape)

        assert len(normalized_shape) == 1

        self.weight = nn.Parameter(torch.ones(normalized_shape))
        self.bias = nn.Parameter(torch.zeros(normalized_shape))
        self.normalized_shape = normalized_shape

    def forward(self, x):
        x = x.contiguous() 
        mu = x.mean(-1, keepdim=True)
        sigma = x.var(-1, keepdim=True, unbiased=False)
        return (x - mu) / torch.sqrt(sigma+1e-5) * self.weight + self.bias
    
class LayerNorm(nn.Module):
    """ Layer Normalization supporting two types: BiasFree and WithBias """
    def __init__(self, dim, LayerNorm_type, out_4d=True):
        super().__init__()
        if LayerNorm_type =='BiasFree':
            self.body = BiasFree_LayerNorm(dim)
        else:
            self.body = WithBias_LayerNorm(dim)
        self.out_4d = out_4d

    def to_3d(self, x):
        # Convert (B, C, H, W) to (B, H*W, C)
        if len(x.shape) == 3:
            return x
        elif len(x.shape) == 4:
            return rearrange(x, 'b c h w -> b (h w) c')
        else:
            raise ValueError("Input must be a 3D or 4D tensor")
    
    def to_4d(self, x, h, w):
        # Convert (B, H*W, C) to (B, C, H, W)
        if len(x.shape) == 4:
            return x
        elif len(x.shape) == 3:
            return rearrange(x, 'b (h w) c -> b c h w', h=h, w=w)
        else:
            raise ValueError("Input must be a 3D or 4D tensor")

    def forward(self, x):
        if self.out_4d:
            h, w = x.shape[-2:]
            return self.to_4d(self.body(self.to_3d(x)), h, w)
        else:
            return self.body(x)

## Reparameterization Block (RepBlock)

In [4]:
class RepConv3(nn.Module):
    def __init__(self, in_channels, out_channels, groups, deploy=False):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.groups = groups
        self.deploy = deploy
        self.reparam = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, groups=groups)
        if not deploy:
            self.conv_3x3 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, groups=groups)
            self.conv_1x1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, groups=groups)
            self.conv_1x3 = nn.Conv2d(in_channels, out_channels, kernel_size=(1, 3), padding=(0, 1), groups=groups)
            self.conv_3x1 = nn.Conv2d(in_channels, out_channels, kernel_size=(3, 1), padding=(1, 0), groups=groups)
            self.conv_1x1_branch = nn.Conv2d(in_channels, in_channels, kernel_size=1, groups=groups, bias=False)
            self.conv_3x3_branch = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, groups=groups, bias=False)
        else:
            self._delete_branches()

    def _delete_branches(self):
        for name in ['conv_3x3','conv_1x1','conv_1x3','conv_3x1', 'conv_1x1_branch', 'conv_3x3_branch']:
            if hasattr(self, name):
                delattr(self, name)

    def fuse(self, delete_branches=True):
        if self.deploy:
            return
        # Extract weights and biases
        conv_3x3_w, conv_3x3_b = self.conv_3x3.weight, self.conv_3x3.bias
        conv_1x1_w, conv_1x1_b = self.conv_1x1.weight, self.conv_1x1.bias
        conv_1x3_w, conv_1x3_b = self.conv_1x3.weight, self.conv_1x3.bias
        conv_3x1_w, conv_3x1_b = self.conv_3x1.weight, self.conv_3x1.bias
        conv_1x1_branch_w, conv_3x3_branch_w = self.conv_1x1_branch.weight, self.conv_3x3_branch.weight
        # Pad the smaller kernels to 3x3
        conv_1x1_w_pad = F.pad(conv_1x1_w, [1, 1, 1, 1])
        conv_1x3_w_pad = F.pad(conv_1x3_w, [0, 0, 1, 1])
        conv_3x1_w_pad = F.pad(conv_3x1_w, [1, 1, 0, 0])
        if self.groups == 1:
            conv_1x1_3x3_w_pad = F.conv2d(conv_3x3_branch_w, conv_1x1_branch_w.permute(1, 0, 2, 3))
        else:
            w_slices = []
            conv_1x1_branch_w_T = conv_1x1_branch_w.permute(1, 0, 2, 3)
            in_channels_per_group = self.in_channels // self.groups
            out_channels_per_group = self.out_channels // self.groups
            for g in range(self.groups):
                # Slice the transposed 1x1 weights for this group's channels
                conv_1x1_branch_w_T_slice = conv_1x1_branch_w_T[:, g*in_channels_per_group:(g+1)*in_channels_per_group, :, :]
                # Slice the 3x3 weights for this group's output channels
                conv_3x3_branch_w_slice = conv_3x3_branch_w[g*out_channels_per_group:(g+1)*out_channels_per_group, :, :, :]
                w_slices.append(F.conv2d(conv_3x3_branch_w_slice, conv_1x1_branch_w_T_slice))
            conv_1x1_3x3_w_pad = torch.cat(w_slices, dim=0)
        # Fuse weights and biases
        conv_w = conv_3x3_w + conv_1x1_w_pad + conv_1x3_w_pad + conv_3x1_w_pad + conv_1x1_3x3_w_pad
        if conv_3x3_b is None:
            conv_3x3_b = torch.zeros(self.out_channels, device=conv_w.device)
        conv_b = conv_3x3_b + conv_1x1_b + conv_1x3_b + conv_3x1_b
        self.reparam.weight.data.copy_(conv_w)
        self.reparam.bias.data.copy_(conv_b)
        # Delete the original branches
        if delete_branches:
            self._delete_branches()
        # Set deploy flag
        self.deploy = True

    def forward(self, x):
        if self.deploy:
            return self.reparam(x)
        else:
            return self.conv_3x3(x) + self.conv_1x1(x) + self.conv_1x3(x) + self.conv_3x1(x) + self.conv_3x3_branch(self.conv_1x1_branch(x))

# # Test repconv
# conv = RepConv3(16, 32, groups=16, deploy=False)
# x = torch.randn(1, 16, 64, 64)
# out1 = conv(x)
# print(f"Before fusion: {count_params(conv)} parameters")
# conv.fuse()
# out2 = conv(x)
# print(f"After fusion: {count_params(conv)} parameters")
# print(torch.allclose(out1, out2, atol=1e-5))

In [5]:
class RepConv5(nn.Module):
    def __init__(self, in_channels, out_channels, groups=1, deploy=False):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.groups = groups
        self.deploy = deploy
        self.reparam = nn.Conv2d(in_channels, out_channels, 5, padding=2, groups=groups)

        if not deploy:
            # Main branches
            self.conv_5x5 = nn.Conv2d(in_channels, out_channels, 5, padding=2, groups=groups)
            self.conv_3x3 = nn.Conv2d(in_channels, out_channels, 3, padding=1, groups=groups)
            self.conv_1x1 = nn.Conv2d(in_channels, out_channels, 1, groups=groups)
            # Asymmetric branches
            self.conv_1x5 = nn.Conv2d(in_channels, out_channels, (1,5), padding=(0,2), groups=groups)
            self.conv_5x1 = nn.Conv2d(in_channels, out_channels, (5,1), padding=(2,0), groups=groups)
            self.conv_1x3 = nn.Conv2d(in_channels, out_channels, (1,3), padding=(0,1), groups=groups)
            self.conv_3x1 = nn.Conv2d(in_channels, out_channels, (3,1), padding=(1,0), groups=groups)
            self.conv_3x5 = nn.Conv2d(in_channels, out_channels, (3,5), padding=(1,2), groups=groups)
            self.conv_5x3 = nn.Conv2d(in_channels, out_channels, (5,3), padding=(2,1), groups=groups)
            # Sequential branch
            self.conv_1x1_branch = nn.Conv2d(in_channels, in_channels, 1, groups=groups, bias=False)
            self.conv_5x5_branch = nn.Conv2d(in_channels, out_channels, 5, padding=2, groups=groups, bias=False)
        else:
            self._delete_branches()

    def _delete_branches(self):
        for name in [
            'conv_5x5','conv_3x3','conv_1x1',
            'conv_1x5','conv_5x1',
            'conv_1x3','conv_3x1',
            'conv_3x5','conv_5x3',
            'conv_1x1_branch','conv_5x5_branch'
        ]:
            if hasattr(self, name):
                delattr(self, name)

    def _pad_to_5x5(self, w):
        _, _, h, w_k = w.shape
        pad_h = 5 - h
        pad_w = 5 - w_k
        pad_top = pad_h // 2
        pad_bottom = pad_h - pad_top
        pad_left = pad_w // 2
        pad_right = pad_w - pad_left
        return F.pad(w, [pad_left, pad_right, pad_top, pad_bottom])

    def fuse(self, delete_branches=True):
        if self.deploy:
            return
        def get_wb(conv):
            return conv.weight, conv.bias if conv.bias is not None else torch.zeros(self.out_channels, device=conv.weight.device)
        W = 0
        B = 0
        # Helper to accumulate
        def add_branch(w, b):
            nonlocal W, B
            W = W + w
            B = B + b
        # Main kernels
        w, b = get_wb(self.conv_5x5)
        add_branch(w, b)
        w, b = get_wb(self.conv_3x3)
        add_branch(self._pad_to_5x5(w), b)
        w, b = get_wb(self.conv_1x1)
        add_branch(self._pad_to_5x5(w), b)
        # Asymmetric
        w, b = get_wb(self.conv_1x5)
        add_branch(self._pad_to_5x5(w), b)
        w, b = get_wb(self.conv_5x1)
        add_branch(self._pad_to_5x5(w), b)
        w, b = get_wb(self.conv_1x3)
        add_branch(self._pad_to_5x5(w), b)
        w, b = get_wb(self.conv_3x1)
        add_branch(self._pad_to_5x5(w), b)
        w, b = get_wb(self.conv_3x5)
        add_branch(self._pad_to_5x5(w), b)
        w, b = get_wb(self.conv_5x3)
        add_branch(self._pad_to_5x5(w), b)
        # Sequential 1x1 -> 5x5
        w1 = self.conv_1x1_branch.weight
        w2 = self.conv_5x5_branch.weight
        if self.groups == 1:
            w_seq = F.conv2d(w2, w1.permute(1,0,2,3))
        else:
            w_slices = []
            w1_T = w1.permute(1,0,2,3)
            icpg = self.in_channels // self.groups
            ocpg = self.out_channels // self.groups
            for g in range(self.groups):
                w1_slice = w1_T[:, g*icpg:(g+1)*icpg]
                w2_slice = w2[g*ocpg:(g+1)*ocpg]
                w_slices.append(F.conv2d(w2_slice, w1_slice))
            w_seq = torch.cat(w_slices, dim=0)
        add_branch(w_seq, torch.zeros(self.out_channels, device=w_seq.device))
        self.reparam.weight.data.copy_(W)
        self.reparam.bias.data.copy_(B)
        if delete_branches:
            self._delete_branches()
        self.deploy = True

    def forward(self, x):
        if self.deploy:
            return self.reparam(x)
        return (
            self.conv_5x5(x)
            + self.conv_3x3(x)
            + self.conv_1x1(x)
            + self.conv_1x5(x)
            + self.conv_5x1(x)
            + self.conv_1x3(x)
            + self.conv_3x1(x)
            + self.conv_3x5(x)
            + self.conv_5x3(x)
            + self.conv_5x5_branch(self.conv_1x1_branch(x))
        )

# # Test RepConv5
# conv = RepConv5(16, 32, groups=16, deploy=False)
# x = torch.randn(1, 16, 64, 64)
# out1 = conv(x)
# print(f"Before fusion: {count_params(conv)} parameters")
# conv.fuse()
# out2 = conv(x)
# print(f"After fusion: {count_params(conv)} parameters")
# print(torch.allclose(out1, out2, atol=1e-5))

In [6]:
class RepConv7(nn.Module):
    def __init__(self, in_channels, out_channels, groups=1, deploy=False):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.groups = groups
        self.deploy = deploy

        self.reparam = nn.Conv2d(in_channels, out_channels, 7, padding=3, groups=groups)

        if not deploy:
            # Main
            self.conv_7x7 = nn.Conv2d(in_channels, out_channels, 7, padding=3, groups=groups)
            # Directional
            self.conv_7x1 = nn.Conv2d(in_channels, out_channels, (7,1), padding=(3,0), groups=groups)
            self.conv_1x7 = nn.Conv2d(in_channels, out_channels, (1,7), padding=(0,3), groups=groups)
            # Mixed large
            self.conv_7x5 = nn.Conv2d(in_channels, out_channels, (7,5), padding=(3,2), groups=groups)
            self.conv_5x7 = nn.Conv2d(in_channels, out_channels, (5,7), padding=(2,3), groups=groups)
            # Mid
            self.conv_5x5 = nn.Conv2d(in_channels, out_channels, 5, padding=2, groups=groups)
            # Small directional
            self.conv_1x5 = nn.Conv2d(in_channels, out_channels, (1,5), padding=(0,2), groups=groups)
            self.conv_5x1 = nn.Conv2d(in_channels, out_channels, (5,1), padding=(2,0), groups=groups)
            # Sequential branch
            self.conv_1x1_branch = nn.Conv2d(in_channels, in_channels, 1, groups=groups, bias=False)
            self.conv_7x7_branch = nn.Conv2d(in_channels, out_channels, 7, padding=3, groups=groups, bias=False)
        else:
            self._delete_branches()

    def _delete_branches(self):
        for name in [
            'conv_7x7',
            'conv_7x1','conv_1x7',
            'conv_7x5','conv_5x7',
            'conv_5x5',
            'conv_1x5','conv_5x1',
            'conv_1x1_branch','conv_7x7_branch'
        ]:
            if hasattr(self, name):
                delattr(self, name)

    def _pad_to_7x7(self, w):
        _, _, h, w_k = w.shape
        pad_h = 7 - h
        pad_w = 7 - w_k
        pad_top = pad_h // 2
        pad_bottom = pad_h - pad_top
        pad_left = pad_w // 2
        pad_right = pad_w - pad_left
        return F.pad(w, [pad_left, pad_right, pad_top, pad_bottom])

    def fuse(self, delete_branches=True):
        if self.deploy:
            return
        def get_wb(conv):
            return conv.weight, conv.bias if conv.bias is not None else torch.zeros(self.out_channels, device=conv.weight.device)
        W = torch.zeros_like(self.reparam.weight)
        B = torch.zeros_like(self.reparam.bias)
        def add_branch(w, b):
            nonlocal W, B
            W += w
            B += b
        # Main
        w, b = get_wb(self.conv_7x7)
        add_branch(w, b)
        # Directional
        for conv in [self.conv_7x1, self.conv_1x7]:
            w, b = get_wb(conv)
            add_branch(self._pad_to_7x7(w), b)
        # Mixed large
        for conv in [self.conv_7x5, self.conv_5x7]:
            w, b = get_wb(conv)
            add_branch(self._pad_to_7x7(w), b)
        # Mid
        w, b = get_wb(self.conv_5x5)
        add_branch(self._pad_to_7x7(w), b)
        # Small directional
        for conv in [self.conv_1x5, self.conv_5x1]:
            w, b = get_wb(conv)
            add_branch(self._pad_to_7x7(w), b)
        # Sequential 1x1 → 7x7
        w1 = self.conv_1x1_branch.weight
        w2 = self.conv_7x7_branch.weight
        if self.groups == 1:
            w_seq = F.conv2d(w2, w1.permute(1,0,2,3))
        else:
            w_slices = []
            w1_T = w1.permute(1,0,2,3)
            icpg = self.in_channels // self.groups
            ocpg = self.out_channels // self.groups
            for g in range(self.groups):
                w1_slice = w1_T[:, g*icpg:(g+1)*icpg]
                w2_slice = w2[g*ocpg:(g+1)*ocpg]
                w_slices.append(F.conv2d(w2_slice, w1_slice))
            w_seq = torch.cat(w_slices, dim=0)
        add_branch(w_seq, torch.zeros(self.out_channels, device=w_seq.device))
        self.reparam.weight.data.copy_(W)
        self.reparam.bias.data.copy_(B)
        if delete_branches:
            self._delete_branches()
        self.deploy = True

    def forward(self, x):
        if self.deploy:
            return self.reparam(x)
        return (
            self.conv_7x7(x)
            + self.conv_7x1(x)
            + self.conv_1x7(x)
            + self.conv_7x5(x)
            + self.conv_5x7(x)
            + self.conv_5x5(x)
            + self.conv_1x5(x)
            + self.conv_5x1(x)
            + self.conv_7x7_branch(self.conv_1x1_branch(x))
        )

# # Test RepConv7
# conv = RepConv7(16, 32, groups=16, deploy=False)
# x = torch.randn(1, 16, 64, 64)
# out1 = conv(x)
# print(f"Before fusion: {count_params(conv)} parameters")
# conv.fuse()
# out2 = conv(x)
# print(f"After fusion: {count_params(conv)} parameters")
# print(torch.allclose(out1, out2, atol=1e-5))

## Attention Block (MonarchAttn)

In [7]:
from monarch_attn import MonarchAttention

@dataclass
class RepAttnConfig:
    dim: int
    num_heads: int = 8
    block_size: int = 16
    num_steps: int = 2
    pad_type: str = "pre"
    impl: str = "torch"
    deploy: bool = False

class RepAttn(nn.Module):
    """ Re-parameterizable Attention Block using MonarchAttention as the core attention mechanism."""
    def __init__(self, dim, num_heads=8, block_size=14, num_steps=1, pad_type="pre", impl="torch", deploy=False):
        super().__init__()
        self.num_heads = num_heads
        self.qkv = nn.Conv2d(dim, dim * 3, kernel_size=1)
        self.monarch_attn = MonarchAttention(
            block_size=block_size,
            num_steps=num_steps,
            pad_type=pad_type,
            impl=impl
        )
        if deploy:
            self.attn_fn = self.monarch_attn
        else:
            self.attn_fn = self.common_attn
        self.proj = nn.Conv2d(dim, dim, kernel_size=1)
        self.deploy = deploy

    def common_attn(self, q, k, v):
        """ Scaled Dot-Product Attention """
        scale = (q.shape[-1]) ** -0.5
        attn = (q @ k.transpose(-2, -1)) * scale
        attn = attn.softmax(dim=-1)
        out = attn @ v
        return out

    @torch.no_grad()
    def fuse(self):
        if not self.deploy:
            self.attn_fn = self.monarch_attn
            self.deploy = True

    def forward(self, x):
        B, C, H, W = x.shape
        qkv = self.qkv(x)
        q, k, v = torch.chunk(qkv, 3, dim=1)
        q = rearrange(q, 'b (head c) h w -> b head c (h w)', head=self.num_heads)
        k = rearrange(k, 'b (head c) h w -> b head c (h w)', head=self.num_heads)
        v = rearrange(v, 'b (head c) h w -> b head c (h w)', head=self.num_heads)
        attn_out = self.attn_fn(q, k, v)
        attn_out = rearrange(attn_out, 'b head c (h w) -> b (head c) h w', head=self.num_heads, h=H, w=W)
        out = self.proj(attn_out)
        return out

# # Test RepAttn
# attn = RepAttn(dim=256, num_heads=8, block_size=14, num_steps=1, pad_type="pre", impl="torch", deploy=False).cuda()
# x = torch.randn(1, 256, 45, 31).cuda()
# out = attn(x)
# print(out.shape)

## Simple ConvFFN

In [8]:
@dataclass
class FFNConfig:
    dim: int
    expansion_factor: int = 1
    deploy: bool = False

In [9]:
class RepFFN(nn.Module):
    def __init__(self, dim, expansion_factor=1, deploy=False):
        super().__init__()
        hidden_features = int(dim * expansion_factor)
        self.project_in = RepConv3(dim, hidden_features, groups=1, deploy=deploy)
        self.dwconv = RepConv3(hidden_features, hidden_features*2, groups=hidden_features, deploy=deploy)
        self.project_out = nn.Conv2d(hidden_features, dim, kernel_size=1)

    @torch.no_grad()
    def fuse(self):
        self.project_in.fuse()
        self.dwconv.fuse()  


    def forward(self, x):
        x = self.project_in(x)
        x1, x2 = self.dwconv(x).chunk(2, dim=1)
        x = F.gelu(x1) * x2
        x = self.project_out(x)
        return x

## Model Blocks

In [10]:
class SkipConnection(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.conv = nn.Conv2d(dim*2, dim, kernel_size=1)

    def forward(self, x1, x2):
        x = torch.cat([x1, x2], dim=1)
        x = self.conv(x)
        return x
    
class RepTransformerBlock(nn.Module):
    def __init__(self, rep_attn_cfg: RepAttnConfig, ffn_cfg: FFNConfig, norm_type='WithBias'):
        super().__init__()
        self.rep_attn = RepAttn(**asdict(rep_attn_cfg))
        self.rep_ffn = RepFFN(**asdict(ffn_cfg))
        self.norm1 = LayerNorm(rep_attn_cfg.dim, norm_type)
        self.norm2 = LayerNorm(rep_attn_cfg.dim, norm_type)

    @torch.no_grad()
    def fuse(self):
        self.rep_attn.fuse()
        self.rep_ffn.fuse()

    def forward(self, x):
        x = x + self.rep_attn(self.norm1(x))
        x = x + self.rep_ffn(self.norm2(x))
        return x
    
class Block(nn.Module):
    def __init__(self, num_block, rep_attn_cfg: RepAttnConfig, ffn_cfg: FFNConfig, norm_type='WithBias'):
        super().__init__()
        self.num_block = num_block
        self.blocks = nn.ModuleList([
            RepTransformerBlock(rep_attn_cfg, ffn_cfg, norm_type) for _ in range(num_block)
        ])

    @torch.no_grad()
    def fuse(self):
        for block in self.blocks:
            block.fuse()

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        return x

In [20]:
class MS2I(nn.Module):
    """ Main model implementation """
    def __init__(self, input_shape=(3, 256, 256), deploy=False, dims=[48, 96, 192, 384], num_blocks=[4, 6, 6, 8], num_heads=[1, 2, 2, 4], bias=True, last_act=None):
        super().__init__()
        assert len(dims) == len(num_blocks) == len(num_heads), "Length of dims, num_blocks and num_heads must be the same"
        self.input_shape = input_shape
        self.deploy = deploy
        self.dims = dims
        self.num_blocks = num_blocks
        self.bias = bias
        self.num_heads = num_heads

        # Extractor
        self.stem = nn.Conv2d(3, dims[0], kernel_size=7, stride=4, padding=3, bias=bias)
        
        # Encoder
        layers = []
        down_convs = []
        for idx in range(len(dims)):
            attn_cfg, ffn_cfg = self.build_cfg(dims[idx], num_heads[idx])
            block = Block(num_blocks[idx], attn_cfg, ffn_cfg, norm_type='WithBias')
            if idx < len(dims) - 1:
                down_convs.append(DownSample(dims[idx]))
            layers.append(block)
        self.bottleneck = layers[-1] # Last encoder layer as bottleneck
        self.encoder = nn.ModuleList(layers[:-1])
        self.downsample = nn.ModuleList(down_convs)
        
        # Decoder
        layers = []
        up_convs = []
        skip_connections = []
        for idx in range(len(dims)-2, -1, -1):
            attn_cfg, ffn_cfg = self.build_cfg(dims[idx], num_heads[idx])
            # print(f"Decoder layer {idx}: shape {l_shape}")
            up_conv = UpSample(dims[idx+1])
            block = Block(num_blocks[idx], attn_cfg, ffn_cfg, norm_type='WithBias')
            layers.append(block)
            up_convs.append(up_conv)
            skip_connections.append(SkipConnection(dims[idx]))
        self.decoder = nn.ModuleList(layers)
        self.up_sample = nn.ModuleList(up_convs)
        self.skip = nn.ModuleList(skip_connections)

        # Head
        self.head = nn.Sequential(
            RepConv3(dims[0], dims[0]//2, 1, deploy=deploy),
            nn.GELU(),
            nn.Conv2d(dims[0]//2, 3, kernel_size=1, bias=bias),
            last_act if last_act is not None else nn.Identity()
        )

    @torch.no_grad()
    def fuse(self):
        for block in self.encoder:
            block.fuse()
        for block in self.decoder:
            block.fuse()
        for conv in self.head:
            if isinstance(conv, RepConv3):
                conv.fuse()
    
    def build_cfg(self, dim, head):
        # RepAttn config
        attn_cfg = RepAttnConfig(
            dim=dim,
            num_heads=head,
            block_size=16,
            num_steps=2,
            pad_type="pre",
            impl="torch",
            deploy=self.deploy
        )
        ## FFN config
        ffn_cfg = FFNConfig(
            dim=dim,
            expansion_factor=1,
        )
        return attn_cfg, ffn_cfg

    def forward(self, x):
        """ 
        x: (B, C, H, W)
        """
        x = self.stem(x)
        # print(f"Extractor output shape: {x.shape}")
        feats = []
        for blk, down in zip(self.encoder, self.downsample):
            x = blk(x)
            feats.append(x)
            x = down(x)
        # for feat in feats:
        #     print(feat.shape)
        x = self.bottleneck(x)
        # print(x.shape)
        # print(len(self.decoder), len(self.up_sample), len(self.skip), len(feats))
        for blk, up, skip in zip(self.decoder, self.up_sample, self.skip):
            x = up(x)
            # print(x.shape, feat.shape)
            cur_feat = feats.pop()
            x = skip(x, cur_feat)
            x = blk(x)
        x = F.interpolate(x, size=self.input_shape[1:], mode='bilinear', align_corners=False)
        x = self.head(x)
        return x

In [49]:
model_cfg = {
    'input_shape': (3, 256, 256),
    'dims': [32, 64, 128, 256],
    'num_blocks': [1, 2, 2, 4],
    'num_heads': [1, 2, 4, 8],
    'bias': True,
    'last_act': None,
    'deploy': False
}

# Test the model
model = MS2I(**model_cfg).to(DEVICE)
x = torch.randn(1, 3, 256, 256)
y = model(x.to(DEVICE))
print(f"Output shape: {y.shape}")

Output shape: torch.Size([1, 3, 256, 256])


In [51]:
from fvcore.nn import FlopCountAnalysis, flop_count_table

print("MS2I Model Parameter Count:")
total, trainable = count_params(model)
print(f"Total: {total:,} | Trainable: {trainable:,}")
print("#" * 50)
print("MS2I Model FLOPs:")
x = torch.randn(1, 3, 256, 256)
flops = FlopCountAnalysis(model, x.to(DEVICE))
print(f"Total FLOPs: {flops.total():,.2f}")
print("#" * 50)
print("Detailed Parameter Count:")
print(flop_count_table(flops))

MS2I Model Parameter Count:
Total: 14,267,411 | Trainable: 14,267,411
##################################################
MS2I Model FLOPs:


Unsupported operator aten::mul encountered 322 time(s)
Unsupported operator aten::mul_ encountered 70 time(s)
Unsupported operator aten::mean encountered 28 time(s)
Unsupported operator aten::var encountered 28 time(s)
Unsupported operator aten::sub encountered 28 time(s)
Unsupported operator aten::add encountered 200 time(s)
Unsupported operator aten::sqrt encountered 28 time(s)
Unsupported operator aten::div encountered 28 time(s)
Unsupported operator aten::pow encountered 14 time(s)
Unsupported operator aten::softmax encountered 14 time(s)
Unsupported operator aten::gelu encountered 15 time(s)
Unsupported operator aten::pixel_unshuffle encountered 3 time(s)
Unsupported operator aten::pixel_shuffle encountered 3 time(s)
The following submodules of the model were never called during the trace of the graph. They may be unused, or they were accessed by direct calls to .forward() or via other python methods. In the latter case they will have zeros for statistics, though their statistics 

Total FLOPs: 2,896,625,664.00
##################################################
Detailed Parameter Count:
| module                            | #parameters or shape   | #flops     |
|:----------------------------------|:-----------------------|:-----------|
| model                             | 14.267M                | 2.897G     |
|  stem                             |  4.736K                |  19.268M   |
|   stem.weight                     |   (32, 3, 7, 7)        |            |
|   stem.bias                       |   (32,)                |            |
|  bottleneck.blocks                |  10.581M               |  0.528G    |
|   bottleneck.blocks.0             |   2.645M               |   0.132G   |
|    bottleneck.blocks.0.rep_attn   |    0.263M              |    17.826M |
|    bottleneck.blocks.0.rep_ffn    |    2.381M              |    0.114G  |
|    bottleneck.blocks.0.norm1.body |    0.512K              |    0       |
|    bottleneck.blocks.0.norm2.body |    0.512K          

In [53]:
# Fuse model
model.fuse()
x = torch.randn(1, 3, 256, 256)
y = model(x.to(DEVICE))
print(f"Output shape after fusion: {y.shape}")

Output shape after fusion: torch.Size([1, 3, 256, 256])


In [54]:
from fvcore.nn import FlopCountAnalysis, flop_count_table

print("MS2I Model Parameter Count (After fusion):")
total, trainable = count_params(model)
print(f"Total: {total:,} | Trainable: {trainable:,}")
print("#" * 50)
print("MS2I Model FLOPs (After fusion):")
x = torch.randn(1, 3, 256, 256)
flops = FlopCountAnalysis(model, x.to(DEVICE))
print(f"Total FLOPs (After fusion): {flops.total():,.2f}")
print("#" * 50)
print("Detailed Parameter Count (After fusion):")
print(flop_count_table(flops))

MS2I Model Parameter Count (After fusion):
Total: 12,017,939 | Trainable: 12,017,939
##################################################
MS2I Model FLOPs (After fusion):


Unsupported operator aten::mul encountered 382 time(s)
Unsupported operator aten::mul_ encountered 70 time(s)
Unsupported operator aten::mean encountered 28 time(s)
Unsupported operator aten::var encountered 28 time(s)
Unsupported operator aten::sub encountered 108 time(s)
Unsupported operator aten::add encountered 186 time(s)
Unsupported operator aten::sqrt encountered 28 time(s)
Unsupported operator aten::div encountered 68 time(s)
Unsupported operator aten::pad encountered 30 time(s)
Unsupported operator aten::where encountered 20 time(s)
Unsupported operator aten::exp encountered 20 time(s)
Unsupported operator aten::sum encountered 50 time(s)
Unsupported operator aten::xlogy encountered 20 time(s)
Unsupported operator aten::mT encountered 10 time(s)
Unsupported operator aten::softmax encountered 24 time(s)
Unsupported operator aten::gelu encountered 15 time(s)
Unsupported operator aten::pixel_unshuffle encountered 3 time(s)
Unsupported operator aten::pow encountered 4 time(s)
Unsu

Total FLOPs (After fusion): 1,573,715,968.00
##################################################
Detailed Parameter Count (After fusion):
| module                            | #parameters or shape   | #flops     |
|:----------------------------------|:-----------------------|:-----------|
| model                             | 12.018M                | 1.574G     |
|  stem                             |  4.736K                |  19.268M   |
|   stem.weight                     |   (32, 3, 7, 7)        |            |
|   stem.bias                       |   (32,)                |            |
|  bottleneck.blocks                |  10.581M               |  0.528G    |
|   bottleneck.blocks.0             |   2.645M               |   0.132G   |
|    bottleneck.blocks.0.rep_attn   |    0.263M              |    17.826M |
|    bottleneck.blocks.0.rep_ffn    |    2.381M              |    0.114G  |
|    bottleneck.blocks.0.norm1.body |    0.512K              |    0       |
|    bottleneck.blocks.0.no

In [37]:
import time

x = torch.randn(1, 3, 256, 256).to('cpu')
model = model.to('cpu')
# Warm-up
with torch.no_grad():
    for _ in range(10):
        _ = model(x)

# Timing
start_time = time.time()
with torch.no_grad():
    for _ in range(100):
        _ = model(x)
end_time = time.time()
avg_time = (end_time - start_time) / 100
print(f"Average inference time: {avg_time:.4f} seconds")

Average inference time: 0.0918 seconds


# Simple Convolution Discriminator

In [55]:
# Conv-BN-LeakyReLU block
class ConvBNLReLU(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=3, stride=1, padding=1, negative_slope=0.2):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel, stride, padding, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.LeakyReLU(negative_slope, inplace=True)
        )

    def forward(self, x):
        return self.block(x)


# Residual Block
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, downsample=False):
        super().__init__()

        stride = 2 if downsample else 1

        self.conv1 = ConvBNLReLU(in_ch, out_ch, stride=stride)
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(out_ch)
        )

        self.act = nn.LeakyReLU(0.2, inplace=True)

        # Skip connection
        if downsample or in_ch != out_ch:
            self.skip = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )
        else:
            self.skip = nn.Identity()

    def forward(self, x):
        identity = self.skip(x)

        out = self.conv1(x)
        out = self.conv2(out)

        out += identity
        out = self.act(out)
        return out

In [56]:
class Discriminator(nn.Module):
    def __init__(self, in_channels=3, base_dim=64):
        super().__init__()

        # Stem
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, base_dim, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(base_dim),
            nn.LeakyReLU(0.2, inplace=True)
        )

        # Residual stages
        self.layer1 = ResBlock(base_dim, base_dim * 2, downsample=True)
        self.layer2 = ResBlock(base_dim * 2, base_dim * 4, downsample=True)
        self.layer3 = ResBlock(base_dim * 4, base_dim * 8, downsample=True)
        self.layer4 = ResBlock(base_dim * 8, base_dim * 8, downsample=False)

        # Patch output head
        self.head = nn.Conv2d(base_dim * 8, 1, kernel_size=4, stride=1, padding=1)

    def forward(self, x):
        x = self.stem(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.head(x)
        return x

In [57]:
discriminator = Discriminator().to(DEVICE)
x = torch.randn(1, 3, 256, 256).to(DEVICE)
out = discriminator(x)
print(f"Discriminator output shape: {out.shape}")

Discriminator output shape: torch.Size([1, 1, 15, 15])


In [58]:
from fvcore.nn import FlopCountAnalysis, flop_count_table

print("Discriminator Model Parameter Count:")
total, trainable = count_params(discriminator)
print(f"Total: {total:,} | Trainable: {trainable:,}")
print("#" * 50)
print("Discriminator Model FLOPs:")
x = torch.randn(1, 3, 256, 256)
flops = FlopCountAnalysis(discriminator, x.to(DEVICE))
print(f"Total FLOPs: {flops.total():,.2f}")
print("#" * 50)
print("Detailed Parameter Count:")
print(flop_count_table(flops))

Unsupported operator aten::add_ encountered 16 time(s)
Unsupported operator aten::leaky_relu_ encountered 9 time(s)


Discriminator Model Parameter Count:
Total: 9,554,305 | Trainable: 9,554,305
##################################################
Discriminator Model FLOPs:
Total FLOPs: 4,099,022,848.00
##################################################
Detailed Parameter Count:
| module                  | #parameters or shape   | #flops     |
|:------------------------|:-----------------------|:-----------|
| model                   | 9.554M                 | 4.099G     |
|  stem                   |  3.2K                  |  55.575M   |
|   stem.0                |   3.072K               |   50.332M  |
|    stem.0.weight        |    (64, 3, 4, 4)       |            |
|   stem.1                |   0.128K               |   5.243M   |
|    stem.1.weight        |    (64,)               |            |
|    stem.1.bias          |    (64,)               |            |
|  layer1                 |  0.23M                 |  0.947G    |
|   layer1.conv1.block    |   73.984K              |   0.305G   |
|    layer1.

# Training Script

In [ ]:
""" 
Implement the training loop + logger + checkpointing + evaluation per epoch + utilities + ... for training the MS2I model.
"""